# Rangkaian neural berulang

Dalam modul sebelumnya, kami telah membincangkan perwakilan semantik yang kaya bagi teks. Seni bina yang telah kami gunakan menangkap makna agregat kata-kata dalam ayat, tetapi ia tidak mengambil kira **susunan** kata-kata, kerana operasi agregasi yang mengikuti penempatan ini menghapuskan maklumat ini daripada teks asal. Oleh kerana model-model ini tidak dapat mewakili susunan kata, mereka tidak boleh menyelesaikan tugas yang lebih kompleks atau samar seperti penjanaan teks atau menjawab soalan.

Untuk menangkap makna urutan teks, kami akan menggunakan seni bina rangkaian neural yang dipanggil **rangkaian neural berulang**, atau RNN. Apabila menggunakan RNN, kami menghantar ayat kami melalui rangkaian satu token pada satu masa, dan rangkaian itu menghasilkan beberapa **keadaan**, yang kemudian kami hantar kembali ke rangkaian bersama token seterusnya.

![Gambar yang menunjukkan contoh penjanaan rangkaian neural berulang.](../../../../../translated_images/ms/rnn.27f5c29c53d727b5.webp)

Diberi urutan input token $X_0,\dots,X_n$, RNN mencipta satu urutan blok rangkaian neural, dan melatih urutan ini secara menyeluruh menggunakan propagasi balik. Setiap blok rangkaian mengambil pasangan $(X_i,S_i)$ sebagai input, dan menghasilkan $S_{i+1}$ sebagai hasil. Keadaan akhir $S_n$ atau output $Y_n$ dimasukkan ke dalam pengelasan linear untuk menghasilkan keputusan. Semua blok rangkaian berkongsi berat yang sama, dan dilatih secara menyeluruh menggunakan satu laluan propagasi balik.

> Rajah di atas menunjukkan rangkaian neural berulang dalam bentuk terbuka (di sebelah kiri), dan dalam perwakilan berulang yang lebih padat (di sebelah kanan). Penting untuk sedar bahawa semua Sel RNN mempunyai **berat yang boleh dikongsi** yang sama.

Oleh kerana vektor keadaan $S_0,\dots,S_n$ dihantar melalui rangkaian, RNN dapat mempelajari pergantungan berurutan antara kata-kata. Contohnya, apabila perkataan *tidak* muncul di mana-mana dalam urutan, ia boleh belajar untuk menafikan unsur-unsur tertentu dalam vektor keadaan.

Di dalamnya, setiap sel RNN mengandungi dua matriks berat: $W_H$ dan $W_I$, serta bias $b$. Pada setiap langkah RNN, diberikan input $X_i$ dan keadaan input $S_i$, keadaan output dikira sebagai $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, di mana $f$ adalah fungsi pengaktifan (seringnya $\tanh$).

> Untuk masalah seperti penjanaan teks (yang akan kami bincangkan dalam unit seterusnya) atau terjemahan mesin, kami juga mahu mendapatkan nilai output pada setiap langkah RNN. Dalam kes ini, terdapat juga matriks lain $W_O$, dan output dikira sebagai $Y_i=f(W_O\times S_i+b_O)$.

Mari kita lihat bagaimana rangkaian neural berulang boleh membantu kita mengklasifikasikan set data berita kita.

> Untuk persekitaran pasir, kita perlu menjalankan sel berikut untuk memastikan perpustakaan yang diperlukan dipasang, dan data dipra-muat. Jika anda menjalankan secara tempatan, anda boleh abaikan sel berikut.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Apabila melatih model besar, peruntukan memori GPU mungkin menjadi masalah. Kami juga mungkin perlu bereksperimen dengan saiz minibatch yang berbeza, supaya data muat ke dalam memori GPU kami, namun latihan cukup pantas. Jika anda menjalankan kod ini pada mesin GPU anda sendiri, anda boleh bereksperimen dengan melaraskan saiz minibatch untuk mempercepatkan latihan.

> **Nota**: Sesetengah versi pemacu NVidia diketahui tidak melepaskan memori selepas melatih model. Kami menjalankan beberapa contoh dalam nota ini, dan ia mungkin menyebabkan memori habis dalam beberapa tetapan, terutamanya jika anda melakukan eksperimen anda sendiri sebagai sebahagian daripada nota yang sama. Jika anda menghadapi beberapa ralat pelik apabila mula melatih model, anda mungkin mahu mulakan semula kernel nota.


In [3]:
batch_size = 16
embed_size = 64

## Pengelasan RNN Mudah

Dalam kes RNN mudah, setiap unit berulang adalah rangkaian linear mudah, yang menerima vektor input dan vektor keadaan, dan menghasilkan vektor keadaan baru. Dalam Keras, ini boleh diwakili oleh lapisan `SimpleRNN`.

Walaupun kita boleh menghantar token yang dikodkan one-hot terus ke lapisan RNN, ini bukan idea yang baik kerana dimensi mereka yang tinggi. Oleh itu, kita akan menggunakan lapisan embedding untuk mengurangkan dimensi vektor perkataan, diikuti oleh lapisan RNN, dan akhirnya pengelasan `Dense`.

> **Nota**: Dalam kes di mana dimensi tidak begitu tinggi, contohnya apabila menggunakan tokenisasi tahap aksara, mungkin masuk akal untuk menghantar token yang dikodkan one-hot terus ke dalam sel RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Nota:** Kami menggunakan lapisan embedding yang tidak dilatih di sini untuk kesederhanaan, tetapi untuk hasil yang lebih baik kami boleh menggunakan lapisan embedding yang telah dilatih menggunakan Word2Vec, seperti yang diterangkan dalam unit sebelumnya. Ia akan menjadi latihan yang baik untuk anda menyesuaikan kod ini untuk berfungsi dengan embedding yang telah dilatih.

Sekarang mari kita latih RNN kita. RNN secara amnya agak sukar untuk dilatih, kerana setelah sel RNN dibuka sepanjang panjang urutan, bilangan lapisan yang terlibat dalam backpropagation sangat besar. Oleh itu kita perlu memilih kadar pembelajaran yang lebih kecil, dan melatih rangkaian pada set data yang lebih besar untuk menghasilkan keputusan yang baik. Ini boleh mengambil masa yang agak lama, jadi penggunaan GPU adalah lebih digalakkan.

Untuk mempercepatkan, kita hanya akan melatih model RNN pada tajuk berita, mengabaikan penerangan. Anda boleh cuba melatih dengan penerangan dan lihat jika anda boleh mendapatkan model untuk dilatih.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Nota** bahawa ketepatan mungkin lebih rendah di sini, kerana kita hanya melatih pada tajuk berita.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Sekarang bahawa kita menggunakan penutupan, kita boleh melatih model pada keseluruhan set data tajuk dan deskripsi.

> **Nota**: Adakah anda perasan bahawa kita telah menggunakan vektorisasi yang dilatih pada tajuk berita, dan bukan pada keseluruhan badan artikel? Potensi, ini boleh menyebabkan beberapa token diabaikan, jadi adalah lebih baik untuk melatih semula vektorisasi tersebut. Walau bagaimanapun, ia mungkin hanya memberi kesan yang sangat kecil, jadi kita akan kekal pada vektorisasi pra-latih sebelumnya demi kesederhanaan.


## LSTM: Ingatan jangka pendek panjang

Salah satu masalah utama RNN adalah **gradien pudar**. RNN boleh menjadi agak panjang, dan mungkin menghadapi kesukaran untuk menghantar semula gradien hingga ke lapisan pertama rangkaian semasa penyebaran balik. Apabila ini berlaku, rangkaian tidak dapat mempelajari hubungan antara token yang jauh. Salah satu cara untuk mengelakkan masalah ini adalah dengan memperkenalkan **pengurusan keadaan secara eksplisit** menggunakan **pintu**. Dua seni bina yang paling biasa yang memperkenalkan pintu adalah **ingatan jangka pendek panjang** (LSTM) dan **unit pintu gerbang* (GRU). Kami akan membincangkan LSTM di sini.

![Imej menunjukkan contoh sel ingatan jangka pendek panjang](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Rangkaian LSTM diatur secara serupa dengan RNN, tetapi terdapat dua keadaan yang dihantar dari lapisan ke lapisan: keadaan sebenar $c$, dan vektor tersembunyi $h$. Pada setiap unit, vektor tersembunyi $h_{t-1}$ digabungkan dengan input $x_t$, dan bersama-sama mereka mengawal apa yang berlaku pada keadaan $c_t$ dan output $h_{t}$ melalui **pintu**. Setiap pintu mempunyai aktivasi sigmoid (output dalam julat $[0,1]$), yang boleh dianggap sebagai penapis bit apabila didarab dengan vektor keadaan. LSTM mempunyai pintu berikut (dari kiri ke kanan pada gambar di atas):
* **pintu lupa** yang menentukan komponen vektor $c_{t-1}$ yang perlu kita lupakan, dan yang perlu diteruskan.
* **pintu input** yang menentukan berapa banyak maklumat daripada vektor input dan vektor tersembunyi sebelumnya harus dimasukkan ke dalam vektor keadaan.
* **pintu output** yang mengambil vektor keadaan baru dan memutuskan komponen mana yang akan digunakan untuk menghasilkan vektor tersembunyi baru $h_t$.

Komponen keadaan $c$ boleh dianggap sebagai bendera yang boleh dihidupkan dan dimatikan. Contohnya, apabila kita menemui nama *Alice* dalam urutan, kita menganggap ia merujuk kepada seorang wanita, dan menaikkan bendera dalam keadaan yang mengatakan kita ada kata nama perempuan dalam ayat itu. Apabila kita seterusnya menemui perkataan *dan Tom*, kita akan menaikkan bendera yang mengatakan kita mempunyai kata nama jamak. Oleh itu, dengan mengawal keadaan kita boleh mengesan sifat tatabahasa ayat tersebut.

> **Nota**: Berikut adalah sumber yang baik untuk memahami dalaman LSTM: [Memahami Rangkaian LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) oleh Christopher Olah.

Walaupun struktur dalaman sel LSTM mungkin kelihatan kompleks, Keras menyembunyikan pelaksanaan ini di dalam lapisan `LSTM`, jadi satu-satunya perkara yang perlu kita lakukan dalam contoh di atas adalah menggantikan lapisan rekuren:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Nota** bahawa melatih LSTM juga agak perlahan, dan anda mungkin tidak nampak peningkatan ketepatan yang banyak pada permulaan latihan. Anda mungkin perlu meneruskan latihan untuk beberapa masa bagi mencapai ketepatan yang baik.


## RNN dwidepan dan berlapis

Dalam contoh-contoh kami setakat ini, rangkaian berulang beroperasi dari permulaan satu urutan sehingga ke akhir. Ini dirasakan semula jadi kepada kami kerana ia mengikuti arah yang sama seperti mana kita membaca atau mendengar ucapan. Walau bagaimanapun, untuk senario yang memerlukan akses rawak kepada urutan input, lebih masuk akal untuk menjalankan pengiraan berulang dalam kedua-dua arah. RNN yang membenarkan pengiraan dalam kedua-dua arah dipanggil RNN **dwidepan**, dan ia boleh dihasilkan dengan membalut lapisan berulang itu dengan lapisan khas `Bidirectonal`.

> **Nota**: Lapisan `Bidirectional` membuat dua salinan lapisan di dalamnya, dan menetapkan sifat `go_backwards` salah satu salinan itu kepada `True`, menjadikannya bergerak ke arah bertentangan sepanjang urutan.

Rangkaian berulang, sama ada searah atau dwidepan, menangkap corak dalam satu urutan, dan menyimpannya ke dalam vektor keadaan atau mengembalikannya sebagai output. Sama seperti rangkaian konvolusi, kita boleh membina satu lagi lapisan berulang mengikuti lapisan pertama untuk menangkap corak tahap lebih tinggi, yang dibina daripada corak tahap rendah yang diekstrak oleh lapisan pertama. Ini membawa kita kepada konsep **RNN berlapis**, yang terdiri daripada dua atau lebih rangkaian berulang, di mana output lapisan sebelumnya dipassing kepada lapisan seterusnya sebagai input.

![Imej menunjukkan RNN long-short-term-memory berlapis](../../../../../translated_images/ms/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Gambar dari [pos hebat ini](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) oleh Fernando López.*

Keras memudahkan pembinaan rangkaian ini, kerana anda hanya perlu menambah lebih banyak lapisan berulang ke dalam model. Untuk semua lapisan kecuali yang terakhir, kita perlu menentukan parameter `return_sequences=True`, kerana kita mahu lapisan itu memulangkan semua keadaan perantaraan, bukan hanya keadaan akhir pengiraan berulang.

Mari bina LSTM dwidepan dua lapisan untuk masalah peng分类 kami.

> **Nota** kod ini sekali lagi mengambil masa yang agak lama untuk disiapkan, tetapi ia memberi kita ketepatan tertinggi yang kami lihat setakat ini. Jadi mungkin berbaloi untuk menunggu dan melihat hasilnya.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN untuk tugas lain

Sehingga kini, kami telah memberi tumpuan kepada menggunakan RNN untuk mengklasifikasikan urutan teks. Tetapi mereka boleh mengendalikan banyak lagi tugas, seperti penjanaan teks dan terjemahan mesin &mdash; kita akan mempertimbangkan tugas-tugas tersebut dalam unit seterusnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
